# GrantScopeAI — Source Access and Validation

## Project Context
Researchers and research-support teams often search grant databases, funder websites, and publication indexes separately. This makes it difficult to see how a topic is funded across regions, which institutions are active, whether research activity is growing, and which funded projects are most comparable to a new concept.

GrantScopeAI consolidates these signals into one exploratory research-intelligence interface.

### Intended User

The primary intended user is a researcher or university research-support professional assessing how to position an early grant concept.

Potential users include:

- Researchers and principal investigators
- University grant offices
- Research-support professionals
- Research strategy teams
- Scientific consultancies

### Primary User Story

As a researcher developing an AI-enabled chemistry or materials proposal, I want to compare my concept with recent funded projects and publication trends so that I can identify relevant funders, refine my positioning, and document the evidence behind my choices.

### Core Decision Supported

GrantScopeAI is intended to help users answer:

Where does a proposed AI-enabled chemistry or materials topic fit within recent funding and publication activity?

### Primary Research Question

How can public grant and publication data help a researcher identify funding patterns, comparable funded projects, and research momentum for an AI-enabled chemistry or materials proposal?

### Supporting Questions

1. How have grant counts and reported award amounts changed over time?
2. Which funders, programmes, countries, and organisations are most active?
3. Which scientific topics appear most frequently in funded projects?
4. Which previously funded projects are most similar to a new proposal concept?
5. Is publication activity increasing or decreasing for selected topics?
6. What data-quality and comparability limitations affect the analysis?

### Initial Scientific Scope

The initial project scope focuses on AI-enabled chemistry and materials research, including:

- Catalysis
- Molecular modelling
- Reaction prediction
- Materials discovery
- Laboratory automation
- Scientific machine learning

The initial analysis period is 2021–2025. Records from 2026 may be included only when coverage is sufficiently complete and clearly labelled as partial-year data.

### Data Source Roles

The project uses three public data sources:

- **CORDIS:** European Union-funded research projects and grant information
- **NSF Award Search:** United States research awards and funding metadata
- **OpenAlex:** Publication activity, research topics, institutions, countries, and citation context

CORDIS and NSF records will be standardised into a common grants dataset. OpenAlex publications will remain in a separate dataset and will be used to provide aggregate research-momentum context.

The project will not force direct row-level matches between grants and publications unless a reliable relationship can be established.

### Planned Project Output

GrantScopeAI will provide:

- Funding-trend exploration
- Analysis of active funders, programmes, organisations, and countries
- Topic-level funding and publication comparisons
- Data-quality reporting
- A keyword-overlap recommendation baseline
- A TF-IDF and cosine-similarity recommender for finding similar funded projects
- A four-page Streamlit decision-support application

### Project Boundaries

GrantScopeAI is an exploratory research-intelligence prototype. It will not:

- Write complete grant proposals
- Determine whether a research idea is genuinely novel
- Predict whether a proposal will receive funding
- Replace official funder eligibility checks
- Establish causal relationships between funding and publication growth
- Compare EUR and USD award totals without a documented conversion method

# Source validation and Raw Dataset Creation
## ### Initial NSF API Test

In [1]:
import json
import os
from datetime import datetime
from pathlib import Path

import pandas as pd
import requests

RAW_DATA_DIR = Path("../data/raw")
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

EXTRACTION_DATE = datetime.now().strftime("%Y-%m-%d")

print(f"Extraction date: {EXTRACTION_DATE}")
print(f"Raw data folder: {RAW_DATA_DIR.resolve()}")

Extraction date: 2026-08-01
Raw data folder: C:\Users\kahau\OneDrive\Documents\Techmeup\Final_Project\data\raw


In [2]:
def inspect_api_response(response, source_name):
    """
    Display basic validation information for an API response.
    """
    print(f"Source: {source_name}")
    print(f"Status code: {response.status_code}")
    print(f"Content type: {response.headers.get('content-type')}")
    print(f"Response size: {len(response.content):,} bytes")

    response.raise_for_status()

In [3]:
NSF_API_URL = "https://api.nsf.gov/services/v1/awards.json"

nsf_params = {
    "keyword": "machine learning chemistry",
    "startDateStart": "01/01/2021",
    "startDateEnd": "12/31/2025",
    "offset": 1,
    "printFields": ",".join([
        "id",
        "title",
        "abstractText",
        "startDate",
        "expDate",
        "awardeeName",
        "awardeeCity",
        "awardeeStateCode",
        "awardeeCountryCode",
        "fundsObligatedAmt",
        "estimatedTotalAmt",
        "fundProgramName",
        "primaryProgram",
        "agency"
    ])
}

nsf_response = requests.get(
    NSF_API_URL,
    params=nsf_params,
    timeout=30
)

inspect_api_response(nsf_response, "NSF Award Search")

Source: NSF Award Search
Status code: 200
Content type: application/json
Response size: 127,271 bytes


In [4]:
nsf_json = nsf_response.json()

print(nsf_json.keys())
print(nsf_json.get("response", {}).keys())

dict_keys(['response'])
dict_keys(['award', 'metadata'])


In [5]:
nsf_awards = nsf_json.get("response", {}).get("award", [])

print(f"Number of awards returned: {len(nsf_awards)}")

nsf_df = pd.DataFrame(nsf_awards)
nsf_df.head()

Number of awards returned: 25


,abstractText,activeAwd,agency,awardAgencyCode,awardee,awardeeAddress,awardeeCity,awardeeCountryCode,awardeeDistrict,awardeeDistrictCode,...,program,progRefCode,publicAccessMandate,startDate,title,transType,ueiNumber,coPDPI,jrnl,publicationResearch
0,Beaches are coastline features that offer econ...,true,NSF,4900,FLORIDA INTERNATIONAL UNIVERSITY,11200 SW 8TH ST,MIAMI,US,26,FL26,...,RET SUPP-Res Exp for Tchr Supp,7218,1,12/15/2025,Collaborative Research: Swash zone dynamics dr...,Standard Grant,Q3KCVK5S9CP1,NaN,NaN,NaN
1,This project aims to serve the national intere...,true,NSF,4900,"LOYOLA UNIVERSITY MARYLAND, INC.",4501 N CHARLES ST,BALTIMORE,US,02,MD02,...,"QUANTUM INFORMATION SCIENCE, Improv Undergrad ...","7203, 8209, 9178",1,12/15/2025,Cornerstones for an Undergraduate Quantum Comp...,Standard Grant,FV5AVEGVTUE4,[Mary L Lowe mlowe@loyola.edu],NaN,NaN
2,"Coral reefs nurture fisheries, protect coastli...",true,NSF,4900,"UNIVERSITY OF CALIFORNIA, LOS ANGELES",10889 WILSHIRE BLVD STE 700,LOS ANGELES,US,36,CA36,...,,,1,12/15/2025,Collaborative Research: BoCP-Implementation: A...,Standard Grant,RN64EPNH8JC6,"[George Perry ghp3@psu.edu, Laura S Weyrich ls...",NaN,NaN
3,"Coral reefs nurture fisheries, protect coastli...",true,NSF,4900,REGENTS OF THE UNIVERSITY OF MICHIGAN,1109 GEDDES AVE STE 3300,ANN ARBOR,US,06,MI06,...,,,1,12/15/2025,Collaborative Research: BoCP-Implementation: A...,Standard Grant,GNJ7BBP73WE9,NaN,NaN,NaN
4,Following the collapse of a cloud core to form...,true,NSF,4900,PRESIDENT AND FELLOWS OF HARVARD COLLEGE,1033 MASSACHUSETTS AVE STE 3,CAMBRIDGE,US,05,MA05,...,LABORATORY ASTROPHYSICS,1205,1,12/15/2025,Survival of Interstellar Organics in Protoplan...,Standard Grant,LN53LCFJFL45,NaN,NaN,NaN


In [26]:
print(sorted(nsf_sample_df.columns.tolist()))

['abstractText', 'activeAwd', 'agency', 'awardAgencyCode', 'awardee', 'awardeeAddress', 'awardeeCity', 'awardeeCountryCode', 'awardeeDistrict', 'awardeeDistrictCode', 'awardeeName', 'awardeePhone', 'awardeeStateCode', 'awardeeZipCode', 'cfdaNumber', 'coPDPI', 'date', 'dirAbbr', 'divAbbr', 'estimatedTotalAmt', 'expDate', 'fundAgencyCode', 'fundProgramName', 'fundsObligated', 'fundsObligatedAmt', 'histAwd', 'id', 'initAmendmentDate', 'jrnl', 'latestAmendmentDate', 'managingPec', 'orgCodeDir', 'orgCodeDiv', 'orgLongName', 'orgLongName2', 'orgUrl', 'parentUeiNumber', 'pdPIName', 'perfAddress', 'perfCity', 'perfCountryCode', 'perfDistrict', 'perfDistrictCode', 'perfLocation', 'perfStateCode', 'perfZipCode', 'pi', 'piEmail', 'piFirstName', 'piId', 'piLastName', 'piMiddeInitial', 'poEmail', 'poName', 'poPhone', 'primaryProgram', 'progEleCode', 'progRefCode', 'program', 'publicAccessMandate', 'publicationResearch', 'startDate', 'title', 'transType', 'ueiNumber']


# NSF info Extraction

In [13]:
NSF_QUERIES = [
    '"machine learning" AND chemistry',
    '"artificial intelligence" AND chemistry',
    '"deep learning" AND chemistry',
    '"machine learning" AND materials',
    '"artificial intelligence" AND materials',
    '"deep learning" AND materials',
    '"machine learning" AND catalysis',
    '"materials informatics"',
    "cheminformatics",
    '"molecular machine learning"',
    '"reaction prediction"',
    '"autonomous laboratory"',
    '"self-driving laboratory"'
]

START_DATE = date(2021, 1, 1)
END_DATE = date(2025, 12, 31)

RAW_DATA_DIR = Path("../data/raw/nsf")
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

EXTRACTION_DATE = datetime.now().strftime("%Y-%m-%d")

print("NSF queries:")
for query in NSF_QUERIES:
    print("-", query)

print("Date range:", START_DATE, "to", END_DATE)
print("Save location:", RAW_DATA_DIR.resolve())

NSF queries:
- "machine learning" AND chemistry
- "artificial intelligence" AND chemistry
- "deep learning" AND chemistry
- "machine learning" AND materials
- "artificial intelligence" AND materials
- "deep learning" AND materials
- "machine learning" AND catalysis
- "materials informatics"
- cheminformatics
- "molecular machine learning"
- "reaction prediction"
- "autonomous laboratory"
- "self-driving laboratory"
Date range: 2021-01-01 to 2025-12-31
Save location: C:\Users\kahau\OneDrive\Documents\Techmeup\Final_Project\data\raw\nsf


In [9]:
NSF_QUERIES = [
    "machine learning chemistry",
    "artificial intelligence chemistry",
    "deep learning chemistry",
    "machine learning materials",
    "artificial intelligence materials",
    "deep learning materials",
    "materials informatics",
    "cheminformatics",
    "molecular machine learning",
    "reaction prediction",
    "machine learning catalysis",
    "autonomous laboratory",
    "self-driving laboratory"
]

In [14]:
def generate_monthly_windows(start_date, end_date):
    """
    Generate monthly date ranges between two dates.
    """
    current_date = date(
        start_date.year,
        start_date.month,
        1
    )

    while current_date <= end_date:
        final_day = monthrange(
            current_date.year,
            current_date.month
        )[1]

        window_end = date(
            current_date.year,
            current_date.month,
            final_day
        )

        window_start = max(current_date, start_date)
        window_end = min(window_end, end_date)

        yield window_start, window_end

        if current_date.month == 12:
            current_date = date(
                current_date.year + 1,
                1,
                1
            )
        else:
            current_date = date(
                current_date.year,
                current_date.month + 1,
                1
            )

In [15]:
def fetch_nsf_awards_for_period(
    start_date,
    end_date,
    keyword= None,
    delay= 0.5
):
    """
    Retrieve all NSF awards whose project start date falls
    within the specified period.
    """
    period_awards = []
    offset = 0
    results_per_page = 25

    while True:
        params = {
            "startDateStart": start_date.strftime("%m/%d/%Y"),
            "startDateEnd": end_date.strftime("%m/%d/%Y"),
            "rpp": results_per_page,
            "offset": offset,
            "printFields": ",".join([
                "id",
                "title",
                "abstractText",
                "date",
                "startDate",
                "expDate",
                "awardeeName",
                "awardeeCity",
                "awardeeStateCode",
                "awardeeCountryCode",
                "fundsObligatedAmt",
                "estimatedTotalAmt",
                "fundProgramName",
                "primaryProgram",
                "agency",
                "piFirstName",
                "piLastName",
                "dirAbbr",
                "divAbbr"
            ])
        }

        if keyword:
            params["keyword"] = keyword

        response = requests.get(
            NSF_API_URL,
            params=params,
            timeout=60
        )

        response.raise_for_status()
        payload = response.json()

        response_data = payload.get("response", {})
        awards = response_data.get("award", []) or []

        if isinstance(awards, dict):
            awards = [awards]

        period_awards.extend(awards)

        print(
            f"{start_date:%Y-%m}: "
            f"page returned {len(awards)}, "
            f"total collected {len(period_awards)}"
        )

        if len(awards) < results_per_page:
            break

        offset += results_per_page
        time.sleep(delay)

    return period_awards

In [16]:
query_test_results = []

for query in NSF_QUERIES:
    test_awards = fetch_nsf_awards_for_period(
        start_date=date(2021, 1, 1),
        end_date=date(2021, 1, 31),
        keyword=query
    )

    query_test_results.append({
        "query": query,
        "records_returned": len(test_awards)
    })

query_test_df = pd.DataFrame(query_test_results)

display(query_test_df)

2021-01: page returned 7, total collected 7
2021-01: page returned 2, total collected 2
2021-01: page returned 1, total collected 1
2021-01: page returned 22, total collected 22
2021-01: page returned 12, total collected 12
2021-01: page returned 6, total collected 6
2021-01: page returned 0, total collected 0
2021-01: page returned 0, total collected 0
2021-01: page returned 0, total collected 0
2021-01: page returned 0, total collected 0
2021-01: page returned 0, total collected 0
2021-01: page returned 0, total collected 0
2021-01: page returned 0, total collected 0


,query,records_returned
0,"""machine learning"" AND chemistry",7
1,"""artificial intelligence"" AND chemistry",2
2,"""deep learning"" AND chemistry",1
3,"""machine learning"" AND materials",22
4,"""artificial intelligence"" AND materials",12
5,"""deep learning"" AND materials",6
6,"""machine learning"" AND catalysis",0
7,"""materials informatics""",0
8,cheminformatics,0
9,"""molecular machine learning""",0


In [17]:
NSF_QUERIES = [
    "machine AND learning",
    "artificial AND intelligence",
    "deep AND learning",
    "materials AND informatics",
    "cheminformatics",
    "molecular AND modeling",
    "molecular AND modelling",
    "reaction AND prediction",
    "computational AND chemistry",
    "data-driven AND chemistry",
    "data-driven AND materials",
    "autonomous AND laboratory",
    "self-driving AND laboratory"
]

In [18]:
query_test_results = []

for query in NSF_QUERIES:
    test_awards = fetch_nsf_awards_for_period(
        start_date=date(2021, 1, 1),
        end_date=date(2021, 1, 31),
        keyword=query
    )

    query_test_results.append({
        "query": query,
        "records_returned": len(test_awards)
    })

query_test_df = pd.DataFrame(query_test_results)

display(query_test_df)

2021-01: page returned 25, total collected 25
2021-01: page returned 25, total collected 50
2021-01: page returned 25, total collected 75
2021-01: page returned 25, total collected 100
2021-01: page returned 2, total collected 102
2021-01: page returned 25, total collected 25
2021-01: page returned 25, total collected 50
2021-01: page returned 8, total collected 58
2021-01: page returned 25, total collected 25
2021-01: page returned 19, total collected 44
2021-01: page returned 1, total collected 1
2021-01: page returned 0, total collected 0
2021-01: page returned 25, total collected 25
2021-01: page returned 3, total collected 28
2021-01: page returned 2, total collected 2
2021-01: page returned 0, total collected 0
2021-01: page returned 15, total collected 15
2021-01: page returned 25, total collected 25
2021-01: page returned 18, total collected 43
2021-01: page returned 25, total collected 25
2021-01: page returned 25, total collected 50
2021-01: page returned 25, total collected 

,query,records_returned
0,machine AND learning,102
1,artificial AND intelligence,58
2,deep AND learning,44
3,materials AND informatics,1
4,cheminformatics,0
5,molecular AND modeling,28
6,molecular AND modelling,2
7,reaction AND prediction,0
8,computational AND chemistry,15
9,data-driven AND chemistry,43


In [ ]:
test_params = {
    "dateStart": "01/01/2021",
    "dateEnd": "01/31/2021",
    "rpp": 25,
    "offset": 0
}

test_response = requests.get(
    NSF_API_URL,
    params=test_params,
    timeout=60
)

test_response.raise_for_status()
test_json = test_response.json()

print("Request URL:")
print(test_response.url)

print("\nMetadata:")
print(test_json["response"]["metadata"])

print(
    "\nRecords returned:",
    len(test_json["response"].get("award", []))
)

Request URL:
https://api.nsf.gov/services/v1/awards.json?dateStart=01%2F01%2F2021&dateEnd=01%2F31%2F2021&rpp=25&offset=0

Metadata:
{'offset': 0, 'rpp': 25, 'totalCount': 473}

Records returned: 25


In [40]:
test_awards = fetch_nsf_awards_for_period(
    start_date=date(2021, 1, 1),
    end_date=date(2021, 1, 31),
    keyword=NSF_QUERY
)

print(f"January 2021 awards retrieved: {len(test_awards):,}")

2021-01: page returned 25, total collected 25
2021-01: page returned 25, total collected 50
2021-01: page returned 25, total collected 75
2021-01: page returned 25, total collected 100
2021-01: page returned 25, total collected 125
2021-01: page returned 25, total collected 150
2021-01: page returned 25, total collected 175
2021-01: page returned 25, total collected 200
2021-01: page returned 25, total collected 225
2021-01: page returned 25, total collected 250
2021-01: page returned 25, total collected 275
2021-01: page returned 25, total collected 300
2021-01: page returned 25, total collected 325
2021-01: page returned 25, total collected 350
2021-01: page returned 25, total collected 375
2021-01: page returned 25, total collected 400
2021-01: page returned 25, total collected 425
2021-01: page returned 25, total collected 450
2021-01: page returned 25, total collected 475
2021-01: page returned 25, total collected 500
2021-01: page returned 25, total collected 525
2021-01: page re

In [ ]:
from calendar import monthrange
from datetime import date, datetime
from pathlib import Path
import time

import pandas as pd
import requests


NSF_API_URL = "https://api.nsf.gov/services/v1/awards.json"

 NSF_QUERIES = [
    '"machine learning" AND chemistry',
    '"artificial intelligence" AND chemistry',
    '"deep learning" AND chemistry',
    '"machine learning" AND materials',
    '"artificial intelligence" AND materials',
    '"deep learning" AND materials',
    '"machine learning" AND catalysis',
    '"materials informatics"',
    'cheminformatics',
    '"molecular machine learning"',
    '"reaction prediction"',
    '"autonomous laboratory"',
    '"self-driving laboratory"'
]


START_DATE = date(2021, 1, 1)
END_DATE = date(2021, 1, 31)

RAW_DATA_DIR = Path("../data/raw/nsf")
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

EXTRACTION_DATE = datetime.now().strftime("%Y-%m-%d")

In [51]:
def fetch_nsf_awards_for_period(
    start_date,
    end_date,
    keyword,
    delay=0.25
):
    """
    Retrieve all NSF awards matching one keyword phrase
    whose project start date falls within the specified period.
    """
    if not isinstance(keyword, str) or not keyword.strip():
        raise ValueError(
            "A non-empty keyword string is required. "
            "This prevents an unrestricted NSF download."
        )

    period_awards = []
    offset = 0
    results_per_page = 25

    while True:
        params = {
            "keyword": keyword,
            "startDateStart": start_date.strftime("%m/%d/%Y"),
            "startDateEnd": end_date.strftime("%m/%d/%Y"),
            "rpp": results_per_page,
            "offset": offset
        }

        response = requests.get(
            NSF_API_URL,
            params=params,
            timeout=60
        )

        response.raise_for_status()
        payload = response.json()

        response_data = payload.get("response", {})
        awards = response_data.get("award", []) or []

        if isinstance(awards, dict):
            awards = [awards]

        period_awards.extend(awards)

        if len(awards) < results_per_page:
            break

        offset += results_per_page

        if offset >= 3000:
            raise RuntimeError(
                f"Query reached the 3,000-record limit: {keyword}"
            )

        time.sleep(delay)

    print(
        f"{start_date:%Y-%m} | "
        f"{keyword}: {len(period_awards)} records"
    )

    return period_awards

Test Query

In [52]:
test_query = NSF_QUERIES[0]

test_awards = fetch_nsf_awards_for_period(
    start_date=START_DATE,
    end_date=END_DATE,
    keyword=test_query
)

print(f"Query tested: {test_query}")
print(f"Records retrieved: {len(test_awards):,}")

2021-01 | "machine learning chemistry": 0 records
Query tested: "machine learning chemistry"
Records retrieved: 0


In [53]:
from datetime import date, datetime
from pathlib import Path
import pandas as pd

START_DATE = date(2021, 1, 1)
END_DATE = date(2025, 12, 31)

RAW_DATA_DIR = Path("../data/raw/nsf")
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

EXTRACTION_DATE = datetime.now().strftime("%Y-%m-%d")

print("Date range:", START_DATE, "to", END_DATE)
print("Number of search queries:", len(NSF_QUERIES))
print("Output folder:", RAW_DATA_DIR.resolve())

Date range: 2021-01-01 to 2025-12-31
Number of search queries: 13
Output folder: C:\Users\kahau\OneDrive\Documents\Techmeup\Final_Project\data\raw\nsf


In [19]:
all_nsf_awards = []

checkpoint_path = (
    RAW_DATA_DIR
    / f"nsf_grantscope_checkpoint_{EXTRACTION_DATE}.csv"
)

total_queries = len(NSF_QUERIES)

for query_number, query in enumerate(
    NSF_QUERIES,
    start=1
):
    print(
        f"\n{'=' * 70}\n"
        f"QUERY {query_number} OF {total_queries}: {query}\n"
        f"{'=' * 70}"
    )

    query_total = 0

    for window_start, window_end in generate_monthly_windows(
        START_DATE,
        END_DATE
    ):
        monthly_awards = fetch_nsf_awards_for_period(
            start_date=window_start,
            end_date=window_end,
            keyword=query
        )

        query_total += len(monthly_awards)

        for award in monthly_awards:
            award_copy = award.copy()

            # Record which search found the award.
            award_copy["_matched_query"] = query

            all_nsf_awards.append(award_copy)

    print(
        f"\nCompleted query {query_number} of "
        f"{total_queries}: {query}"
    )
    print(f"Records returned: {query_total:,}")

    # Save progress after each complete query.
    checkpoint_df = pd.DataFrame(all_nsf_awards)

    checkpoint_df.to_csv(
        checkpoint_path,
        index=False
    )

    print(
        f"Checkpoint saved: "
        f"{len(checkpoint_df):,} cumulative rows"
    )

print(
    "\nExtraction complete."
    f"\nRows before deduplication: "
    f"{len(all_nsf_awards):,}"
)


QUERY 1 OF 13: machine AND learning
2021-01: page returned 25, total collected 25
2021-01: page returned 25, total collected 50
2021-01: page returned 25, total collected 75
2021-01: page returned 25, total collected 100
2021-01: page returned 2, total collected 102
2021-02: page returned 25, total collected 25
2021-02: page returned 25, total collected 50
2021-02: page returned 18, total collected 68
2021-03: page returned 25, total collected 25
2021-03: page returned 25, total collected 50
2021-03: page returned 25, total collected 75
2021-03: page returned 11, total collected 86
2021-04: page returned 25, total collected 25
2021-04: page returned 25, total collected 50
2021-04: page returned 25, total collected 75
2021-04: page returned 3, total collected 78
2021-05: page returned 25, total collected 25
2021-05: page returned 25, total collected 50
2021-05: page returned 25, total collected 75
2021-05: page returned 25, total collected 100
2021-05: page returned 10, total collected

In [28]:

nsf_raw_df=pd.DataFrame(all_nsf_awards)

In [29]:
nsf_raw_df

,abstractText,activeAwd,agency,awardAgencyCode,awardee,awardeeAddress,awardeeCity,awardeeCountryCode,awardeeDistrict,awardeeDistrictCode,...,startDate,title,transType,ueiNumber,_matched_query,piMiddeInitial,coPDPI,awdSpAttnCode,awdSpAttnDesc,arraAmount
0,Composite structures have increasingly emerged...,false,NSF,4900,VIRGINIA POLYTECHNIC INSTITUTE & STATE UNIVERSITY,300 TURNER ST NW,BLACKSBURG,US,09,VA09,...,01/15/2021,Ultra-high Precision Assembly of Aerospace Com...,Standard Grant,QDE5UHE5XD16,machine AND learning,NaN,NaN,NaN,NaN,NaN
1,National efforts to digitize natural history c...,false,NSF,4900,UNIVERSITY OF FLORIDA,1523 UNION RD RM 207,GAINESVILLE,US,03,FL03,...,01/15/2021,Collaborative Research: CIBR: Leaping the Spec...,Standard Grant,NNFQH1JAPEP3,machine AND learning,P,NaN,NaN,NaN,NaN
2,In an effort to support decision making by gov...,false,NSF,4900,THE JOHNS HOPKINS UNIVERSITY,3400 N CHARLES ST,BALTIMORE,US,07,MD07,...,01/15/2021,RAPID: Real-time Forecasting of COVID-19 risk ...,Standard Grant,FTMTDMBR29C7,machine AND learning,M,NaN,NaN,NaN,NaN
3,COVID-19 disproportionately affects the low-wa...,false,NSF,4900,WAYNE STATE UNIVERSITY,5700 CASS AVE STE 4900,DETROIT,US,13,MI13,...,01/15/2021,SCC-CIVIC-PG Track A: Leveraging AI-assist Mic...,Standard Grant,M6K6NTJ2MNE5,machine AND learning,NaN,"[Daniel Grosu dgrosu@wayne.edu, Tierra Bills t...",NaN,NaN,NaN
4,The broader impact/commercial potential of thi...,false,NSF,4900,WAYNE STATE UNIVERSITY,5700 CASS AVE STE 4900,DETROIT,US,13,MI13,...,01/15/2021,I-Corps: AI-enabled automation intelligence s...,Standard Grant,M6K6NTJ2MNE5,machine AND learning,NaN,[Murat Yildirim murat@wayne.edu],NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33120,Robotic systems that must operate in hazardous...,true,NSF,4900,THE UNIVERSITY CORPORATION,18111 NORDHOFF ST,NORTHRIDGE,US,32,CA32,...,11/01/2025,Collaborative Research: CyberTraining: Impleme...,Standard Grant,LAGNHMC58DF3,self-driving AND laboratory,NaN,NaN,NaN,NaN,NaN
33121,Robotic systems that must operate in hazardous...,true,NSF,4900,UNIVERSITY OF WISCONSIN SYSTEM,21 N PARK ST STE 6301,MADISON,US,02,WI02,...,11/01/2025,Collaborative Research: CyberTraining: Impleme...,Standard Grant,LCLSJAGTNZQ7,self-driving AND laboratory,NaN,"[Radu Serban serban@engr.wisc.edu, Luning Bakk...",NaN,NaN,NaN
33122,Robotic systems that must operate in hazardous...,true,NSF,4900,CAL POLY POMONA FOUNDATION INC,3801 W TEMPLE AVE,POMONA,US,35,CA35,...,11/01/2025,Collaborative Research: CyberTraining: Impleme...,Standard Grant,JMGMMM7BMBT6,self-driving AND laboratory,NaN,NaN,NaN,NaN,NaN
33123,Robotic systems that must operate in hazardous...,true,NSF,4900,CSU FULLERTON AUXILIARY SERVICES CORPORATION,1121 N STATE COLLEGE BLVD,FULLERTON,US,45,CA45,...,11/01/2025,Collaborative Research: CyberTraining: Impleme...,Standard Grant,VQ5WK498QDC6,self-driving AND laboratory,NaN,NaN,NaN,NaN,NaN


In [23]:
Query_df.columns


Index(['abstractText', 'activeAwd', 'agency', 'awardAgencyCode', 'awardee',
       'awardeeAddress', 'awardeeCity', 'awardeeCountryCode',
       'awardeeDistrict', 'awardeeDistrictCode', 'awardeeName', 'awardeePhone',
       'awardeeStateCode', 'awardeeZipCode', 'cfdaNumber', 'date', 'dirAbbr',
       'divAbbr', 'estimatedTotalAmt', 'expDate', 'fundAgencyCode',
       'fundProgramName', 'fundsObligated', 'fundsObligatedAmt', 'histAwd',
       'id', 'initAmendmentDate', 'jrnl', 'latestAmendmentDate', 'managingPec',
       'orgCodeDir', 'orgCodeDiv', 'orgLongName', 'orgLongName2', 'orgUrl',
       'parentUeiNumber', 'pdPIName', 'perfAddress', 'perfCity',
       'perfCountryCode', 'perfDistrict', 'perfDistrictCode', 'perfLocation',
       'perfStateCode', 'perfZipCode', 'pi', 'piEmail', 'piFirstName', 'piId',
       'piLastName', 'poEmail', 'poName', 'poPhone', 'primaryProgram',
       'progEleCode', 'program', 'progRefCode', 'projectOutComesReport',
       'publicAccessMandate', 'publica

In [2]:
from pathlib import Path

RAW_DATA_DIR = Path("../Data/Raw_Data")
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Saving files to:")
print(RAW_DATA_DIR.resolve())

Saving files to:
C:\Users\kahau\OneDrive\Documents\Techmeup\Final_Project\GrantScopeAI\Data\Raw_Data


In [7]:
from pathlib import Path
from datetime import datetime

print("Current notebook location:")
print(Path.cwd())

SEARCH_ROOT = Path(
    r"C:\Users\kahau\OneDrive\Documents\Techmeup"
)

data_extensions = {".csv", ".jsonl", ".parquet"}

found_files = []

for file in SEARCH_ROOT.rglob("*"):
    if not file.is_file():
        continue

    filename = file.name.lower()

    if (
        file.suffix.lower() in data_extensions
        and (
            "nsf" in filename
            or "grantscope" in filename
            or "checkpoint" in filename
        )
    ):
        found_files.append(file)

found_files = sorted(
    set(found_files),
    key=lambda file: file.stat().st_size,
    reverse=True
)

print(f"\nPotential NSF files found: {len(found_files)}\n")

for file in found_files:
    size_mb = file.stat().st_size / 1_000_000
    modified = datetime.fromtimestamp(
        file.stat().st_mtime
    )

    print(f"{size_mb:,.2f} MB | {modified}")
    print(file)
    print()

Current notebook location:
c:\Users\kahau\OneDrive\Documents\Techmeup\Final_Project\GrantScopeAI\Notebooks

Potential NSF files found: 2

300.18 MB | 2026-08-01 13:12:32.377770
C:\Users\kahau\OneDrive\Documents\Techmeup\Final_Project\data\raw\nsf\nsf_grantscope_raw_2021_2025_2026-08-01.csv

300.18 MB | 2026-08-01 12:55:07.874020
C:\Users\kahau\OneDrive\Documents\Techmeup\Final_Project\data\raw\nsf\nsf_grantscope_checkpoint_2026-08-01.csv



In [8]:
import pandas as pd

csv_files = [
    file
    for file in found_files
    if file.suffix.lower() == ".csv"
]

if not csv_files:
    raise FileNotFoundError(
        "No matching CSV was found."
    )

largest_csv = max(
    csv_files,
    key=lambda file: file.stat().st_size
)

print("Loading:")
print(largest_csv)

nsf_raw_df = pd.read_csv(
    largest_csv,
    low_memory=False
)

print(
    f"Loaded: {nsf_raw_df.shape[0]:,} rows × "
    f"{nsf_raw_df.shape[1]:,} columns"
)

Loading:
C:\Users\kahau\OneDrive\Documents\Techmeup\Final_Project\data\raw\nsf\nsf_grantscope_raw_2021_2025_2026-08-01.csv
Loaded: 33,125 rows × 70 columns


In [9]:
from pathlib import Path
import shutil

current_file = Path(
    r"C:\Users\kahau\OneDrive\Documents\Techmeup\Final_Project\data\raw\nsf"
    r"\nsf_grantscope_raw_2021_2025_2026-08-01.csv"
)

target_folder = Path(
    r"C:\Users\kahau\OneDrive\Documents\Techmeup\Final_Project"
    r"\GrantScopeAI\Data\Raw_Data"
)

target_folder.mkdir(parents=True, exist_ok=True)

target_file = target_folder / current_file.name

shutil.move(current_file, target_file)

print("Moved to:")
print(target_file)
print("File exists:", target_file.exists())

Moved to:
C:\Users\kahau\OneDrive\Documents\Techmeup\Final_Project\GrantScopeAI\Data\Raw_Data\nsf_grantscope_raw_2021_2025_2026-08-01.csv
File exists: True


In [10]:
import pandas as pd

nsf_raw_df = pd.read_csv(
    target_file,
    low_memory=False
)

print(nsf_raw_df.shape)

(33125, 70)
